In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.16 Chemical Equilibrium and the Saha Equation

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.16",
    title="Chemical Equilibrium and the Saha Equation",
    blurb="What the free energies were for: reactions. The law of mass "
    "action discovered twice — once by a numerical minimizer walking "
    "down G, once by chemical potentials balancing exactly — the van 't "
    "Hoff slope measured with its enthalpy correction, and then Saha's "
    "equation applied to hydrogen itself: why the Sun's surface is "
    "neutral, why the Balmer lines crown the A stars, and why the "
    "universe turned transparent at three thousand kelvin instead of "
    "a hundred and fifty thousand.",
    difficulty="intermediate",
    estimate="110–140 min",
)

## Notebook overview

[§5.7](potentials-legendre-maxwell.ipynb) built the free energies and
promised they were the machinery of *equilibrium*; this notebook
spends them on the equilibria chemistry and astrophysics actually
care about — reactions, where particles change identity. The
governing principle is one line: at fixed $T$ and $V$, matter
rearranges until the free energy is minimal, which happens exactly
when the **chemical potentials balance** across the reaction. From
that single condition follows the law of mass action, the van 't
Hoff temperature dependence, and — the crown jewel — **Saha's
equation** {cite}`saha1920` for the "reaction" that made modern
astrophysics possible:
$\mathrm{H} \rightleftharpoons \mathrm{p} + \mathrm{e}^-$.

We do it with the course's standing double-entry bookkeeping: every
equilibrium is computed *twice*, once by brute numerical minimization
of the free energy (the machinery of
[§0.13](../00-foundations/optimization.ipynb), pointed at
thermodynamics) and once by the analytic mass-action condition, and
the two must agree to many digits. Then three applications, each a
famous number: the solar photosphere's one-in-ten-thousand
ionization, the Balmer lines peaking in the A stars, and the cosmic
recombination at $z \approx 1400$ that released the CMB of
[§4.9](../04-special-relativity/relativistic-optics.ipynb). Nolting
{cite}`nolting6` is the companion text.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**Equilibrium is a chemical-potential balance.** For a reaction like
$\mathrm{A_2} \rightleftharpoons 2\mathrm{A}$ at fixed $T, V$, the
Helmholtz free energy $F(N_{\rm A})$ changes by
$(2\mu_{\rm A} - \mu_{\rm A_2})\,dN$ when one molecule dissociates;
the minimum therefore sits exactly where

```{math}
:label: eq-ce-mu
2\,\mu_{\rm A} \;=\; \mu_{\rm A_2},
```

and with the ideal-gas chemical potential
$\mu = k_B T \ln(n \lambda^3) - \varepsilon_{\rm int}$ (the
[§5.8](partition-function.ipynb) partition function differentiated,
with $\lambda$ the thermal wavelength and $\varepsilon_{\rm int}$
the internal binding), {eq}`eq-ce-mu` exponentiates into the **law of
mass action**: the density combination $n_{\rm A}^2/n_{\rm A_2}$
equals a function of temperature alone,
$K(T) = (\lambda_{\rm A_2}^3/\lambda_{\rm A}^6)\,
e^{-\varepsilon_0/k_B T}$, whose logarithm's slope against $1/T$ —
van 't Hoff's relation — measures the reaction's *enthalpy*:
$\varepsilon_0$ **plus** the $\tfrac32 k_B T$ of translational
bookkeeping hiding in the $\lambda^3$ prefactors. Exercise 1 fits
that slope and finds the correction, because bond energy and
enthalpy are not the same number.

**Saha: ionization is a chemical reaction.** Apply the same balance
to $\mathrm{H} \rightleftharpoons \mathrm{p} + \mathrm{e}^-$ with
ionization energy $\chi = 13.6\ \mathrm{eV}$, and (spin weights
$2 \times 1 / 2 = 1$) the electron's tiny mass puts *its* thermal
wavelength in charge:

```{math}
:label: eq-ce-saha
\frac{n_{\rm e}\, n_{\rm p}}{n_{\rm H}}
\;=\;
\left(\frac{2\pi m_e k_B T}{h^2}\right)^{3/2}
e^{-\chi/k_B T}
\;\equiv\; S(T).
```

For a pure-hydrogen gas of total density $n$ and ionized fraction
$x$ (so $n_e = n_p = xn$, $n_{\rm H} = (1-x)n$), this closes into
$x^2/(1-x) = S(T)/n$ — a quadratic, solved in one line.

**The famous surprise.** Setting $k_B T = \chi$ gives
$T = 158{,}000\ \mathrm{K}$, yet hydrogen ionizes at temperatures
ten to forty times lower. The reason is *entropy*: the prefactor
$S(T)/n = (n\lambda_e^3)^{-1}\,e^{-\chi/k_B T}$ carries the number
of free-electron states per atom, and at ordinary densities
$1/(n \lambda_e^3)$ is enormous — the Boltzmann penalty
$e^{-\chi/k_B T}$ only needs to climb to *one part in that huge
phase-space factor* for ionization to win. Ionization happens at
$k_B T \approx \chi/11$ in the photosphere and $\chi/42$ in the
early universe, and the difference between those two numbers is
nothing but density. Every application below is this one sentence,
quantified.

## Setup

SI throughout, constants from `scipy.constants`. The Saha machinery
is three short functions; everything downstream is applications.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import c, eV, h, hbar, k
from scipy.constants import m_e
from scipy.constants import u as amu
from scipy.optimize import brentq, minimize_scalar

from ecp import validate

CHI_H = 13.6057 * eV  # hydrogen ionization energy
E_2 = 10.2041 * eV  # the n = 2 excitation energy (Balmer's lower level)


def lambda3(m, T):
    """Cubed thermal de Broglie wavelength λ³ = (h²/2πmk_BT)^(3/2).

    The quantum volume per particle that every ideal-gas chemical
    potential μ = k_BT ln(nλ³) carries; masses in kg, T in kelvin.

    Parameters
    ----------
    m : float
        Particle mass.
    T : float
        Temperature.

    Returns
    -------
    float
        λ³ in m³.
    """
    return (h**2 / (2.0 * np.pi * m * k * T)) ** 1.5


def saha_S(T):
    """The Saha function S(T) = (2πm_e k_B T/h²)^(3/2) e^(−χ/k_BT).

    The right-hand side of eq-ce-saha for hydrogen: the density of
    available free-electron states discounted by the ionization
    Boltzmann factor.

    Parameters
    ----------
    T : float or numpy.ndarray
        Temperature in kelvin.

    Returns
    -------
    float or numpy.ndarray
        S(T) in m⁻³.
    """
    return (2.0 * np.pi * m_e * k * T / h**2) ** 1.5 * np.exp(-CHI_H / (k * T))


def ion_fraction(T, n):
    """Ionized fraction x of pure hydrogen at temperature T, density n.

    Solves the Saha closure x²/(1−x) = S(T)/n — a quadratic with one
    physical root — for a gas of total hydrogen density n.

    Parameters
    ----------
    T : float or numpy.ndarray
        Temperature in kelvin.
    n : float or numpy.ndarray
        Total hydrogen number density in m⁻³.

    Returns
    -------
    float or numpy.ndarray
        The ionized fraction x ∈ (0, 1).
    """
    r = saha_S(T) / n
    return (-r + np.sqrt(r * r + 4.0 * r)) / 2.0

## Exercise 1 — The law of mass action, discovered twice

A model dissociation $\mathrm{A_2} \rightleftharpoons 2\mathrm{A}$:
atoms of mass $1\,\mathrm{u}$, binding energy
$\varepsilon_0 = 4.5\ \mathrm{eV}$ (hydrogen-molecule-like), total
atom density $10^{24}\ \mathrm{m^{-3}}$, internal structure ignored.

**Part a)** At $T = 3000\ \mathrm{K}$, find the dissociated fraction
$f$ two independent ways: minimize the total Helmholtz free energy
$F(f)$ over $f$ with `scipy.optimize.minimize_scalar` (the
[§0.13](../00-foundations/optimization.ipynb) machinery pointed at
thermodynamics), and solve the analytic mass-action condition with
`scipy.optimize.brentq`. Verify the two agree to `rtol=1e-4`, and
verify {eq}`eq-ce-mu` *at* the minimizer: $2\mu_{\rm A} =
\mu_{\rm A_2}$ to a part in $10^{6}$ — the minimum of $F$ *is* the
chemical-potential balance.

**Part b)** Van 't Hoff, with its fine print. Fit (with
`numpy.polyfit`) the slope of $\ln K$ against $1/T$ over
$T = 2500$–$4000\ \mathrm{K}$. The textbook reading says the slope
is $-\varepsilon_0/k_B$; verify the measured slope is $9\%$ steeper,
and that it matches $-(\varepsilon_0 + \tfrac32 k_B \bar T)/k_B$ to
`rtol=5e-3`: the $T^{3/2}$ prefactor of $K(T)$ contributes the
translational enthalpy, and a plot of $\ln K$ measures the
*enthalpy* of reaction, not the bare bond energy. Chemistry's
tables know this; now our fit does too.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([f_min]),
    np.array([f_an]),
    "the free-energy minimizer and the analytic law of mass action "
    "find the same dissociated fraction: the law, discovered twice",
    rtol=1e-4,
)
validate.check(
    mu_gap < 1e-6,
    "and at the minimum the chemical potentials balance exactly, "
    "2μ_A = μ_A₂: minimizing F IS eq-ce-mu",
    f"relative imbalance {mu_gap:.1e}",
)
validate.close(
    np.array([vh_slope]),
    np.array([vh_expected]),
    "van 't Hoff's slope measures the ENTHALPY, ε₀ + (3/2)k_B T̄ — "
    "9% steeper than the bare bond energy, the T^(3/2) prefactor's "
    "translational contribution",
    rtol=5e-3,
)
validate.check(
    abs(vh_slope) > abs(E_BIND / k) * 1.05,
    "(and indeed visibly steeper than −ε₀/k_B alone: bond energy and "
    "reaction enthalpy are different numbers)",
    f"slope/( −ε₀/k_B) = {vh_slope / (-E_BIND / k):.4f}",
)

## Exercise 2 — Saha and the neutral Sun

Now the reaction that matters most:
$\mathrm{H} \rightleftharpoons \mathrm{p} + \mathrm{e}^-$, in the
model of a pure-hydrogen gas at the solar photosphere's density
$n = 1.2\times10^{23}\ \mathrm{m^{-3}}$.

**Part a)** Verify the photosphere is *neutral*: at
$T = 5800\ \mathrm{K}$ the ionized fraction is
$x = 1.16\times10^{-4}$ (`rtol=2e-2`) — one atom in ten thousand —
even though the Sun is a ball of "ionized plasma" in every
textbook's opening sentence. (It is, deeper down.)

**Part b)** Find where ionization actually happens: verify (with
`scipy.optimize.brentq`) $x = \tfrac12$ at
$T = 14{,}190\ \mathrm{K}$ (`rtol=1e-2`) — which is
$k_B T = \chi/11$, *eleven times* cooler than the naive
$k_B T = \chi$ estimate of $158{,}000\ \mathrm{K}$. Verify the
entropic bookkeeping behind it: at that temperature the phase-space
factor $1/(n\lambda_e^3)$ exceeds $10^{4}$ — the Boltzmann penalty
only has to reach one part in *that* for ionization to win. Verify
also the transition's width: $x$ runs from $0.1$ to $0.9$ across
$10{,}860 \to 18{,}130\ \mathrm{K}$ (`rtol=1e-2` each), a sharp
switch on the scale of $\chi/k_B$ but a gentle one on the scale of
the temperature itself.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([x_5800]),
    np.array([1.16e-4]),
    "the solar photosphere is NEUTRAL: one hydrogen atom in ten "
    "thousand is ionized at 5800 K",
    rtol=2e-2,
)
validate.close(
    np.array([T_half, T_10, T_90]),
    np.array([14190.0, 10860.0, 18130.0]),
    "half-ionization arrives at 14,190 K — kT = χ/11, eleven times "
    "cooler than the naive energy estimate — with the 10–90% switch "
    "spanning 10,860 to 18,130 K",
    rtol=1e-2,
)
validate.check(
    phase_factor > 1e4,
    "because the phase-space factor 1/(nλ_e³) exceeds 10⁴: the "
    "Boltzmann penalty need only reach one part in THAT — ionization "
    "is an entropy story",
    f"1/(nλ_e³) = {phase_factor:.1e}",
)

## Exercise 3 — Why the Balmer lines crown the A stars

The Balmer absorption lines — the visible-light fingerprint of
hydrogen — need atoms that are *neutral* (Saha) yet *excited to*
$n = 2$ (Boltzmann, $E_2 = 10.2\ \mathrm{eV}$). Those two demands
pull in opposite directions, and their product explains a
century-old observational fact: hydrogen lines are weak in cool M
stars, strongest in A stars, and weak again in hot O stars — even
though every one of those stars is mostly hydrogen.

**Part a)** Model the line strength as
$(1 - x(T))\, e^{-E_2/k_B T}$ (neutral fraction times the $n = 2$
Boltzmann factor) at the line-forming density
$n = 10^{20}\ \mathrm{m^{-3}}$ (the tenuous upper photosphere where
absorption lines are made — the density matters, and we state it).
Maximize with `scipy.optimize.minimize_scalar` and verify the peak
sits at $T = 10{,}030\ \mathrm{K}$ (`rtol=1e-2`) — squarely in the
A-star range, where hydrogen lines are indeed observed strongest.

**Part b)** Verify the tug-of-war is genuine: at $7000\ \mathrm{K}$
(a K/G star) the strength is below $25\%$ of the peak because too
few atoms reach $n = 2$, and at $16{,}000\ \mathrm{K}$ (a B star)
it is below $25\%$ again because Saha has taken the atoms
themselves. One curve, and the Harvard spectral sequence's central
puzzle — solved by Payne with exactly this calculation in 1925 —
falls out.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([T_peak]),
    np.array([10030.0]),
    "the Balmer strength peaks at 10,030 K — the A stars, exactly "
    "where a century of stellar spectra puts the strongest hydrogen "
    "lines",
    rtol=1e-2,
)
validate.check(
    s_7000 < 0.25 and s_16000 < 0.25,
    "and the peak is a genuine tug-of-war: a quarter strength or "
    "less on BOTH flanks — too cold for excitation, too hot for "
    "atoms",
    f"relative strengths {s_7000:.3f} (7000 K), {s_16000:.3f} " f"(16,000 K)",
)

## Exercise 4 — Recombination: the universe turns transparent

The grandest Saha application of all. The early universe is a
hydrogen plasma tied to the CMB photon bath of
[§7.14](../07-quantum-statistical-mechanics/photon-gas-planck.ipynb);
as it expands and cools, {eq}`eq-ce-saha` decides when the electrons
finally bind and the photons fly free — the moment whose light *is*
the CMB that [§4.9](../04-special-relativity/relativistic-optics.ipynb)
measured us moving through.

**Part a)** The only inputs are the baryon-to-photon ratio
$\eta = 6.1\times10^{-10}$ and today's CMB temperature
$T_0 = 2.7255\ \mathrm{K}$: the baryon density at temperature $T$
is $n_b = \eta\,n_\gamma(T)$ with
$n_\gamma = (2\zeta(3)/\pi^2)(k_B T/\hbar c)^3$ photons per unit
volume. Verify half-recombination at $T = 3762\ \mathrm{K}$
(`rtol=1e-2`), i.e. redshift $z = T/T_0 - 1 = 1379$ (within
$[1300, 1450]$) — from two measured numbers and one equation, the
epoch of the CMB's release.

**Part b)** The entropy story, at its extreme: verify
$k_B T_{\rm rec} = \chi/42$ — recombination waits until the
temperature is *forty-two times* below the ionization scale, because
there are $1/\eta \sim 10^9$ photons per baryon and the Wien tail
keeps ionizing until the Boltzmann penalty beats *that*. Verify also
the transition's redshift width: $x$ falls from $0.9$ to $0.1$
between $z = 1481$ and $z = 1261$ (`rtol=1e-2` each) — about two
hundred in redshift, the finite thickness of the last-scattering
"surface" whose imprint every CMB map carries.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([T_rec]),
    np.array([3762.0]),
    "half-recombination at 3762 K — redshift 1379 — from two "
    "measured numbers and one equation: the epoch of the CMB's "
    "release, computed",
    rtol=1e-2,
)
validate.check(
    1300.0 < z_rec < 1450.0 and 41.0 < chi_ratio < 43.0,
    "at k_B T = χ/42: a billion photons per baryon hold the plasma "
    "ionized until forty-two times below the ionization scale — the "
    "entropy argument at its cosmic extreme",
    f"z = {z_rec:.0f}, χ/k_BT = {chi_ratio:.1f}",
)
validate.close(
    np.array([z_90, z_10]),
    np.array([1481.0, 1261.0]),
    "with the 0.9 → 0.1 fall spanning Δz ≈ 220: the last-scattering "
    "surface has thickness, and every CMB map carries it",
    rtol=1e-2,
)

## Notebook summary

- The law of mass action was discovered twice and agreed with
  itself: the free-energy minimizer's dissociated fraction matched
  the analytic mass-action root to $10^{-4}$, with
  $2\mu_{\rm A} = \mu_{\rm A_2}$ holding to $10^{-6}$ at the
  minimum.
- Van 't Hoff's slope came out $9\%$ steeper than $-\varepsilon_0 /
  k_B$ and matched $-(\varepsilon_0 + \tfrac32 k_B\bar T)/k_B$ to
  $0.3\%$: $\ln K$ measures enthalpy, not bare bond energy.
- Saha made the Sun's surface neutral ($x = 1.16\times10^{-4}$ at
  $5800\ \mathrm{K}$) and put half-ionization at $\chi/11 k_B$ —
  an entropy result, priced by the $10^4$ phase-space factor
  $1/(n\lambda_e^3)$.
- The Balmer strength — Saha fighting Boltzmann — peaked at
  $10{,}030\ \mathrm{K}$ at the stated line-forming density: the A
  stars, as the Harvard sequence and Payne's 1925 thesis have it.
- And two measured numbers ($\eta$, $T_0$) plus one equation put
  the universe's transparency at $z = 1379$, $k_B T = \chi/42$,
  with a last-scattering thickness of $\Delta z \approx 220$.

## Outlook

- **The same equation, in silicon.** Replace $\chi$ by a band gap
  and $m_e$ by effective masses and {eq}`eq-ce-saha` becomes
  [§7.13](../07-quantum-statistical-mechanics/semiconductor-statistics.ipynb)'s
  law of mass action $np = n_i^2$: ionizing a hydrogen atom and
  exciting an electron-hole pair are the one calculation wearing
  two costumes.
- **Beyond equilibrium.** Real cosmic recombination lags Saha
  slightly (the Peebles three-level treatment) because the
  ground-state photons re-ionize their neighbors; equilibrium gets
  the epoch right to a few percent, and the residual is a
  non-equilibrium story in the spirit of
  [§5.11](taste-of-nonequilibrium.ipynb).
- **Stellar interiors.** Saha zones of partial ionization drive the
  opacity bumps that make Cepheids pulse — the $\kappa$-mechanism:
  this notebook's curve, oscillating a star.
- **Payne's thesis.** The 1925 dissertation that applied Saha to
  the Harvard spectra proved the stars are mostly hydrogen — against
  the era's consensus — and remains a model of an equation changing
  what everyone believed the universe is made of.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()